# Part 2 — Multi-omics state and inferred activities

This notebook is pure data preparation: it turns CPTAC LUAD matrices into paired
tumour-normal contrasts, then infers per-patient transcription-factor and kinase
activities from them. Nothing here touches the graph — the results are written to
disk as `rna_contrast`, `tf_activity`, `kinase_activity`, and a mutation matrix,
which `03` loads to build the patient layers.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

import cptac
import decoupler as dc

import uc1_common as uc

np.random.seed(uc.SEED)

## Load CPTAC LUAD

Transcriptomics, phosphoproteomics, and somatic mutations. The next section
inspects value ranges before choosing any transformation.

In [ ]:
luad = cptac.Luad()
rna_raw = luad.get_transcriptomics('bcm').copy()
pp_raw = luad.get_phosphoproteomics('umich').copy()
soms = luad.get_somatic_mutation('harmonized').copy()
print({'rna_raw_shape': rna_raw.shape, 'phospho_raw_shape': pp_raw.shape, 'somatic_rows': len(soms)})

## Tumour-normal contrasts

CPTAC processed matrices may already be centred or carry negative values, so we
inspect ranges first and use **paired differences** unless a positive raw scale is
clearly justified — recording the choice in the saved tables.

In [ ]:
def flatten_axis(values) -> pd.Index:
    if isinstance(values, pd.MultiIndex):
        flat = ['_'.join(str(x) for x in item if pd.notna(x) and str(x) != 'nan')
                for item in values.to_flat_index()]
        return pd.Index(flat, dtype='object')
    return pd.Index(values.astype(str), dtype='object')



def summarize_numeric_matrix(df: pd.DataFrame, label: str) -> pd.Series:
    arr = df.to_numpy(dtype=float)
    finite = arr[np.isfinite(arr)]
    return pd.Series({
        'label': label, 'n_rows': df.shape[0], 'n_cols': df.shape[1], 'n_values': arr.size,
        'n_finite': int(np.isfinite(arr).sum()), 'finite_fraction': float(np.isfinite(arr).mean()),
        'min': float(np.nanmin(arr)), 'max': float(np.nanmax(arr)), 'mean': float(np.nanmean(arr)),
        'median': float(np.nanmedian(arr)), 'n_negative': int((arr < 0).sum()), 'n_zero': int((arr == 0).sum()),
        'nan_fraction': float(np.isnan(arr).mean()), 'inf_fraction': float(np.isinf(arr).mean()),
        'finite_mean_abs': float(np.mean(np.abs(finite))) if finite.size else np.nan,
    })


def build_matched_pairs(sample_ids: pd.Index) -> pd.DataFrame:
    normals = {s[:-2]: s for s in sample_ids if s.endswith('.N')}
    matched = [t for t in sample_ids if not t.endswith('.N') and t in normals]
    pairs = pd.DataFrame({'tumor': matched, 'normal': [normals[t] for t in matched]}, index=matched)
    pairs.index.name = 'patient_id'
    return pairs


def choose_contrast_method(df: pd.DataFrame, label: str) -> tuple[str, str]:
    s = summarize_numeric_matrix(df, label)
    if bool(s['min'] > 0 and s['n_negative'] == 0):
        return 'difference', f'{label}: all values positive, but the processed CPTAC scale is not confirmed as raw abundance; paired difference used conservatively.'
    return 'difference', f'{label}: non-positive values present, so log2 fold-change is unsafe; paired difference used.'


def compute_paired_contrast(df: pd.DataFrame, pairs: pd.DataFrame, method: str) -> pd.DataFrame:
    tumor, normal = df.loc[pairs['tumor']].copy(), df.loc[pairs['normal']].copy()
    tumor.index = normal.index = pairs.index
    if method == 'log2fc':
        return (np.log2(tumor + 1.0) - np.log2(normal + 1.0)).T
    if method == 'difference':
        return (tumor - normal).T
    raise ValueError(f'Unknown contrast method: {method}')

In [ ]:
rna_sf = rna_raw.copy()
rna_sf.index = flatten_axis(rna_sf.index)
rna_cols = rna_sf.columns
rna_sf.columns = (rna_cols.get_level_values(0) if isinstance(rna_cols, pd.MultiIndex)
                  else rna_cols).astype(str)

# Keep the phospho column MultiIndex; the contrast collapses to `{gene}_{site}`
# keys that match the OmniPath enzyme-substrate format.
pp_sf = pp_raw.copy()
pp_sf.index = flatten_axis(pp_sf.index)

matched_pairs = build_matched_pairs(rna_sf.index)
assert len(matched_pairs) > 0, 'No matched tumor-normal RNA pairs were found.'

rna_method, rna_reason = choose_contrast_method(rna_sf, 'transcriptomics')
pp_method, pp_reason = choose_contrast_method(pp_sf, 'phosphoproteomics')

rna_contrast = compute_paired_contrast(rna_sf, matched_pairs, rna_method).groupby(level=0).mean()
rna_contrast.columns = rna_contrast.columns.astype(str)

pp_contrast = compute_paired_contrast(pp_sf, matched_pairs, pp_method)
if isinstance(pp_contrast.index, pd.MultiIndex):
    pp_contrast = pp_contrast.groupby(level=[0, 1]).mean()
    pp_contrast.index = pd.Index(['_'.join(str(x) for x in idx if pd.notna(x) and str(x) != 'nan')
                                  for idx in pp_contrast.index.to_flat_index()])
else:
    pp_contrast = pp_contrast.groupby(level=0).mean()
pp_contrast.columns = pp_contrast.columns.astype(str)

assert not np.isinf(rna_contrast.to_numpy(dtype=float)).any(), 'Infinite values in RNA contrast.'
assert not np.isinf(pp_contrast.to_numpy(dtype=float)).any(), 'Infinite values in phospho contrast.'

contrast_diagnostics = pd.DataFrame([
    summarize_numeric_matrix(rna_contrast, 'rna_contrast_features_by_patient'),
    summarize_numeric_matrix(pp_contrast, 'phospho_contrast_features_by_patient'),
])
contrast_diagnostics.to_csv(uc.TABLES / 'contrast_diagnostics.csv', index=False)
pd.DataFrame({'data_type': ['transcriptomics', 'phosphoproteomics'], 'method': [rna_method, pp_method],
             'justification': [rna_reason, pp_reason]}).to_csv(uc.TABLES / 'contrast_methods.csv', index=False)

rna_contrast.to_parquet(uc.RNA_CONTRAST)
print(rna_reason)
print(pp_reason)
contrast_diagnostics

### Missing-data handling

Missing values are never dropped: they flow into the diagnostic tables and are
imputed with `0.0` only at the decoupler ULM step, encoding *no observed
differential signal for that feature in that patient*.

In [ ]:
activity_input_diagnostics = pd.DataFrame({
    'matrix': ['rna_contrast', 'phospho_contrast'],
    'nan_fraction_before_fill': [
        float(np.isnan(rna_contrast.to_numpy(dtype=float)).mean()),
        float(np.isnan(pp_contrast.to_numpy(dtype=float)).mean()),
    ],
})
activity_input_diagnostics.to_csv(uc.TABLES / 'activity_input_missingness.csv', index=False)
activity_input_diagnostics

## TF and kinase activity inference

TF activities come from the RNA contrast via CollecTRI regulons; kinase
activities from the phosphosite contrast via OmniPath enzyme-substrate links.
Both are inference layers, not direct measurements.

In [ ]:
collectri = dc.op.collectri()
shared_tf = sorted(set(collectri['target'].astype(str)) & set(rna_contrast.index.astype(str)))
assert shared_tf, 'No shared symbols between rna_contrast and CollecTRI; check RNA feature normalization.'
tf_es, _ = dc.mt.ulm(rna_contrast.fillna(0.0).T, net=collectri)

psp_cache = uc.CACHE / 'omnipath_enz_sub.tsv'
if psp_cache.exists():
    psp = pd.read_csv(psp_cache, sep='\t')
else:
    psp = pd.read_csv(uc.PSP_URL, sep='\t')
    psp.to_csv(psp_cache, sep='\t', index=False)

psp_kin = psp[psp['modification'].eq('phosphorylation')].copy()
psp_kin['target'] = (psp_kin['substrate_genesymbol'].astype(str) + '_'
                     + psp_kin['residue_type'].astype(str) + psp_kin['residue_offset'].astype(str))
psp_net = (psp_kin[['enzyme_genesymbol', 'target']].rename(columns={'enzyme_genesymbol': 'source'})
           .drop_duplicates().assign(weight=1.0))

shared_phospho = sorted(set(pp_contrast.index.astype(str)) & set(psp_net['target'].astype(str)))
assert shared_phospho, 'No shared phosphosites between pp_contrast and OmniPath enzyme-substrate; check phospho normalization.'

kin_es, kin_tmin = None, None
for tmin in uc.ULM_TMIN_CANDIDATES:
    try:
        kin_es, _ = dc.mt.ulm(pp_contrast.fillna(0.0).T, net=psp_net, tmin=tmin)
        kin_tmin = tmin
        break
    except AssertionError as exc:
        kin_error = str(exc)
assert kin_es is not None, f'Kinase inference failed for all tmin. Last error: {kin_error}'

assert np.isfinite(tf_es.to_numpy(dtype=float)).all(), 'Non-finite TF activities.'
assert np.isfinite(kin_es.to_numpy(dtype=float)).all(), 'Non-finite kinase activities.'

pd.DataFrame([
    summarize_numeric_matrix(tf_es.T, 'tf_activity_features_by_patient'),
    summarize_numeric_matrix(kin_es.T, 'kinase_activity_features_by_patient'),
]).to_csv(uc.TABLES / 'activity_diagnostics.csv', index=False)

tf_es.to_parquet(uc.TF_ES)
kin_es.to_parquet(uc.KIN_ES)
print({'tf_activity_shape': tf_es.shape, 'kinase_activity_shape': kin_es.shape, 'kinase_ulm_tmin_used': kin_tmin})

## Mutation burden

The somatic-mutation matrix (protein-altering variants) drives the mutation-burden
cohort split in `03`. We build it here, where CPTAC is loaded, and persist it.

In [ ]:
protein_altering = {'Missense_Mutation', 'Nonsense_Mutation', 'Frame_Shift_Del', 'Frame_Shift_Ins'}
mut_mat = (
    soms[soms['Mutation'].isin(protein_altering)][['Gene', 'Tumor_Sample_Barcode']]
    .drop_duplicates().assign(mutated=1)
    .pivot(index='Gene', columns='Tumor_Sample_Barcode', values='mutated').fillna(0)
)
mut_mat.columns = mut_mat.columns.astype(str).str.replace('_T', '', regex=False)
mut_mat.to_parquet(uc.MUT_MAT)
print(f'mutation matrix: {mut_mat.shape[0]:,} genes x {mut_mat.shape[1]:,} samples -> {uc.MUT_MAT.name}')